# Data Exploration

Goal: understand the structure of available datasets and decide which files are useful for the prediction pipeline.

## Datasets
- `vct_2025`: All VCT 2025 events (Kickoff, Stage 1, Stage 2, Masters Bangkok, Masters Toronto, Champions Paris)
- `vct_historic`: VCT 2021-2026 historical data

Source: Kaggle (piyush86kumar, ryanluong1)

In [2]:
import os
import pandas as pd
from glob import glob

## Setup
Moving working directory to project root so all paths resolve correctly.

In [3]:
# making sure we're in the project root (sometimes were in notebooks/ or vlr-api/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "vlr-api":
    os.chdir("champions-prediction-vlrggapi")

ROOT = os.getcwd()
print(ROOT)

c:\Users\Davi\Desktop\codes\vlr-api\champions-prediction-vlrggapi


In [4]:
from src.config.constants import (
    VCT_2025_PATH,
    VCT_HISTORIC_PATH,
    MASTERS_BANGKOK_2025_PATH,
    MASTERS_TORONTO_2025_PATH,
    CHAMPIONS_2025_PATH
)

## 1. Match Results — `matches.csv`
Checking structure of match results from vct_2025. Looking for: teams, scores, winner, date, tournament stage.

In [5]:

# lets see the structure of the data
df = pd.read_csv(os.path.join(MASTERS_BANGKOK_2025_PATH, "matches.csv"))

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      17 non-null     str  
 1   match_id  17 non-null     int64
 2   time      17 non-null     str  
 3   team1     17 non-null     str  
 4   score1    17 non-null     int64
 5   team2     17 non-null     str  
 6   score2    17 non-null     int64
 7   score     17 non-null     str  
 8   winner    17 non-null     str  
 9   status    17 non-null     str  
 10  week      17 non-null     str  
 11  stage     17 non-null     str  
dtypes: int64(3), str(9)
memory usage: 3.2 KB


(int64 for a 1 digit number is a overkill, future change definetely)

In [6]:
# seeing the data granularity
df.head()

,date,match_id,time,team1,score1,team2,score2,score,winner,status,week,stage
0,"Thu, February 20, 2025",448598,3:30 PM,EDward Gaming,2,Team Liquid,0,2-0,EDward Gaming,Completed,Round 1,Swiss Stage
1,"Thu, February 20, 2025",448600,6:30 PM,DRX,2,Sentinels,0,2-0,DRX,Completed,Round 1,Swiss Stage
2,"Fri, February 21, 2025",448599,3:30 PM,Team Vitality,2,T1,0,2-0,Team Vitality,Completed,Round 1,Swiss Stage
3,"Fri, February 21, 2025",448597,5:55 PM,G2 Esports,2,Trace Esports,0,2-0,G2 Esports,Completed,Round 1,Swiss Stage
4,"Sat, February 22, 2025",449000,3:30 PM,DRX,1,Team Vitality,2,1-2,Team Vitality,Completed,Round 2 (1-0),Swiss Stage


## 2. Historic Dataset — `vct_historic`
Checking if historic data (2021-2024) can complement vct_2025.

In [7]:
# vct_historic has no matches.csv — checking what match-related files are available
path_historic = os.path.join(VCT_HISTORIC_PATH, "vct_2024/matches")
files = os.listdir(path_historic)
print("\n".join(files))

draft_phase.csv
eco_rounds.csv
eco_stats.csv
kills.csv
kills_stats.csv
maps_played.csv
maps_scores.csv
overview.csv
rounds_kills.csv
scores.csv
team_mapping.csv
win_loss_methods_count.csv
win_loss_methods_round_number.csv


In [8]:
df_scores = pd.read_csv(os.path.join(VCT_HISTORIC_PATH, "vct_2024/matches/scores.csv"))
print(df_scores.shape)
df_scores.head()

(434, 9)


,Tournament,Stage,Match Type,Match Name,Team A,Team B,Team A Score,Team B Score,Match Result
0,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Gen.G,Sentinels,2,0,Gen.G won
1,Valorant Champions 2024,Group Stage,Opening (B),FunPlus Phoenix vs Team Heretics,FunPlus Phoenix,Team Heretics,1,2,Team Heretics won
2,Valorant Champions 2024,Group Stage,Opening (A),DRX vs KRÜ Esports,DRX,KRÜ Esports,2,1,DRX won
3,Valorant Champions 2024,Group Stage,Opening (A),FNATIC vs Bilibili Gaming,FNATIC,Bilibili Gaming,2,0,FNATIC won
4,Valorant Champions 2024,Group Stage,Opening (C),LEVIATÁN vs Talon Esports,LEVIATÁN,Talon Esports,2,0,LEVIATÁN won


In [9]:
df_overview = pd.read_csv(os.path.join(VCT_HISTORIC_PATH, "vct_2024/matches/overview.csv"))
print(df_overview.shape)
df_overview.head()

(46152, 21)


,Tournament,Stage,Match Type,Match Name,Map,Player,Team,Agents,Rating,Average Combat Score,...,Deaths,Assists,Kills - Deaths (KD),"Kill, Assist, Trade, Survive %",Average Damage Per Round,Headshot %,First Kills,First Deaths,Kills - Deaths (FKD),Side
0,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.60,305.0,...,12.0,4.0,12.0,81%,188.0,31%,3.0,4.0,-1.0,both
1,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.71,364.0,...,5.0,2.0,7.0,89%,216.0,37%,2.0,2.0,0.0,attack
2,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.52,261.0,...,7.0,2.0,5.0,75%,168.0,24%,1.0,2.0,-1.0,defend
3,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,Meteor,Gen.G,killjoy,1.23,221.0,...,10.0,4.0,7.0,76%,140.0,26%,1.0,4.0,-3.0,both
4,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,Meteor,Gen.G,killjoy,1.16,229.0,...,4.0,1.0,4.0,78%,127.0,35%,1.0,1.0,0.0,attack


### Findings
- No date column anywhere in vct_historic
- Without dates, Elo and time-based features are impossible
- **Decision: park vct_historic, use vct_2025 only**

## 3. Date Consistency Check
Confirming all vct_2025 events have consistent date columns.

In [10]:

# search for any csv in vct_historic that contains a date-related column
for filepath in glob(os.path.join(VCT_HISTORIC_PATH, "**", "*.csv"), recursive=True):
    df_temp = pd.read_csv(filepath, nrows=1)
    cols = [c.lower() for c in df_temp.columns]
    if any("date" in c for c in cols):
        print(filepath)
        print(df_temp.columns.tolist())
        print()

In [11]:
for filepath in glob(os.path.join(VCT_2025_PATH, "**", "*.csv"), recursive=True):
    if "matches.csv" in filepath:
        df_temp = pd.read_csv(filepath, nrows=1)
        print(os.path.relpath(filepath, VCT_2025_PATH))
        print(df_temp["date"].values[0])
        print()

Valorant Champions 2025_csvs\matches.csv
Fri, September 12, 2025

Valorant Masters Bangkok 2025_csvs\matches.csv
Thu, February 20, 2025

Valorant Masters Toronto 2025_csvs\matches.csv
Sat, June 7, 2025

VCT 2025 Americas Kickoff_csvs\matches.csv
Fri, January 17, 2025

VCT 2025 Americas Stage 1_csvs\matches.csv
Sat, March 22, 2025

VCT 2025 Americas Stage 2_csvs\matches.csv
Sat, July 19, 2025

VCT 2025 China Kickoff_csvs\matches.csv
Sat, January 11, 2025

VCT 2025 China Stage 1_csvs\matches.csv
Thu, March 13, 2025

VCT 2025 China Stage 2_csvs\matches.csv
Thu, July 3, 2025

VCT 2025 EMEA Kickoff_csvs\matches.csv
Wed, January 15, 2025

VCT 2025 EMEA Stage 1_csvs\matches.csv
Wed, March 26, 2025

VCT 2025 EMEA Stage 2_csvs\matches.csv
Wed, July 16, 2025

VCT 2025 Pacific Kickoff_csvs\matches.csv
Sat, January 18, 2025

VCT 2025 Pacific Stage 1_csvs\matches.csv
Sat, March 22, 2025

VCT 2025 Pacific Stage 2_csvs\matches.csv
Tue, July 15, 2025



In [12]:
df_player = pd.read_csv(os.path.join(MASTERS_BANGKOK_2025_PATH, "player_stats.csv"))
print(df_player.shape)
df_player.head()

(41, 25)


,player,player_name,team,player_id,agents,agents_count,rounds,rating,acs,kd_ratio,...,fdpr,hs_percent,cl_percent,clutches,k_max,kills,deaths,assists,first_kills,first_deaths
0,CHICHOO,CHICHOO,EDG,15559,"['Viper', 'Vyse', 'Cypher', 'Omen']",4,313,1.23,235.6,1.27,...,0.07,28%,22%,11/51,36,271,213,87,24,22
1,nAts,nAts,TL,457,"['Viper', 'Chamber', 'Cypher']",3,155,1.21,257.6,1.32,...,0.12,26%,13%,3/23,25,141,107,29,14,18
2,trent,trent,G2,15500,"['Tejo', 'Sova']",2,360,1.21,220.0,1.21,...,0.09,25%,29%,14/48,26,274,226,116,31,31
3,Derke,Derke,VIT,5022,"['Raze', 'Jett', 'Yoru']",3,225,1.18,257.2,1.20,...,0.20,22%,31%,5/16,32,204,170,38,46,45
4,iZu,iZu,T1,29833,"['Tejo', 'Fade', 'Cypher', 'Yoru']",4,453,1.13,207.1,1.24,...,0.09,30%,31%,19/61,21,341,276,96,48,40


In [13]:
df_maps = pd.read_csv(f"{MASTERS_BANGKOK_2025_PATH}/maps_stats.csv")
print(df_maps.shape)
df_maps.head()

(7, 4)


,map_name,times_played,attack_win_percent,defense_win_percent
0,Lotus,9,50%,50%
1,Bind,7,46%,54%
2,Split,7,51%,49%
3,Haven,6,60%,40%
4,Pearl,5,44%,56%


In [14]:
df_detailed = pd.read_csv(f"{MASTERS_BANGKOK_2025_PATH}/detailed_matches_player_stats.csv")
print(df_detailed.shape)
df_detailed.head()

(600, 26)


,match_id,event_name,event_stage,match_date,team1,team2,score_overall,player_name,player_id,player_team,...,a,kd_diff,kast,adr,hs_percent,fk,fd,fk_fd_diff,map_name,map_winner
0,448598,Valorant Masters Bangkok 2025,Swiss Stage: \n\t\t\t\t\t\tRound 1,2025-02-20 05:00:00,EDward Gaming,Team Liquid,2 - 0,CHICHOO,15559.0,EDward Gaming,...,15.0,19.0,74%,172.0,35%,3.0,4.0,-1.0,NaN,NaN
1,448598,Valorant Masters Bangkok 2025,Swiss Stage: \n\t\t\t\t\t\tRound 1,2025-02-20 05:00:00,EDward Gaming,Team Liquid,2 - 0,ZmjjKK,3520.0,EDward Gaming,...,11.0,5.0,72%,167.0,26%,14.0,7.0,7.0,NaN,NaN
2,448598,Valorant Masters Bangkok 2025,Swiss Stage: \n\t\t\t\t\t\tRound 1,2025-02-20 05:00:00,EDward Gaming,Team Liquid,2 - 0,nobody,3017.0,EDward Gaming,...,11.0,-3.0,74%,148.0,36%,5.0,8.0,-3.0,NaN,NaN
3,448598,Valorant Masters Bangkok 2025,Swiss Stage: \n\t\t\t\t\t\tRound 1,2025-02-20 05:00:00,EDward Gaming,Team Liquid,2 - 0,S1Mon,13768.0,EDward Gaming,...,18.0,-4.0,64%,108.0,32%,2.0,2.0,0.0,NaN,NaN
4,448598,Valorant Masters Bangkok 2025,Swiss Stage: \n\t\t\t\t\t\tRound 1,2025-02-20 05:00:00,EDward Gaming,Team Liquid,2 - 0,Smoggy,4742.0,EDward Gaming,...,21.0,-8.0,70%,97.0,33%,6.0,2.0,4.0,NaN,NaN


### Findings
- All 15 events have dates ranging from January to September 2025
- Date format is consistent across events

In [15]:
df.columns.tolist()

['date',
 'match_id',
 'time',
 'team1',
 'score1',
 'team2',
 'score2',
 'score',
 'winner',
 'status',
 'week',
 'stage']

## 4. Player Stats — `detailed_matches_player_stats.csv`
Loading all player stats across all 2025 events to build team-level features.

In [20]:
dfs = []
for filepath in glob.glob(os.path.join(VCT_2025_PATH, "**", "detailed_matches_player_stats.csv"), recursive=True):
    df = pd.read_csv(filepath)
    event_name = os.path.basename(os.path.dirname(filepath))
    df["event"] = event_name
    dfs.append(df)

df_players = pd.concat(dfs, ignore_index=True)
print(df_players.shape)
print(df_players.columns.tolist())

(17792, 27)
['match_id', 'event_name', 'event_stage', 'match_date', 'team1', 'team2', 'score_overall', 'player_name', 'player_id', 'player_team', 'stat_type', 'agent', 'rating', 'acs', 'k', 'd', 'a', 'kd_diff', 'kast', 'adr', 'hs_percent', 'fk', 'fd', 'fk_fd_diff', 'map_name', 'map_winner', 'event']


In [17]:
print(df_players["stat_type"].unique())
print(df_players["match_date"].head())

<ArrowStringArray>
['overall', 'map']
Length: 2, dtype: str
0    2025-09-12 09:00:00
1    2025-09-12 09:00:00
2    2025-09-12 09:00:00
3    2025-09-12 09:00:00
4    2025-09-12 09:00:00
Name: match_date, dtype: str


In [18]:
df_overall = df_players[df_players["stat_type"] == "overall"].copy()
df_overall["match_date"] = pd.to_datetime(df_overall["match_date"])
print(df_overall.shape)
print(df_overall[["match_date", "player_team", "acs", "kast", "adr"]].head())

(5032, 27)
           match_date player_team    acs kast    adr
0 2025-09-12 09:00:00   Paper Rex  283.0  88%  188.0
1 2025-09-12 09:00:00   Paper Rex  268.0  78%  165.0
2 2025-09-12 09:00:00   Paper Rex  286.0  78%  183.0
3 2025-09-12 09:00:00   Paper Rex  158.0  73%  107.0
4 2025-09-12 09:00:00   Paper Rex  128.0  65%   89.0


In [19]:
print(df_overall["player_team"].unique())

<ArrowStringArray>
[                                       'Paper Rex',
                                    'Xi Lai Gaming',
                                           'GIANTX',
                                        'Sentinels',
                                              'NRG',
                                    'EDward Gaming',
                                      'Team Liquid',
                                              'DRX',
                             'Dragon Ranger Gaming',
                                               'T1',
                                       'G2 Esports',
                                    'Team Heretics',
 'Guangzhou Huadu Bilibili Gaming(Bilibili Gaming)',
                                             'MIBR',
                                   'Rex Regum Qeon',
                                           'FNATIC',
                                    'Team Vitality',
                                    'Trace Esports',
                           

### Findings
- 17,792 rows total across all 2025 events
- `stat_type` has two values: `overall` (per match) and `map` (per map) — we use `overall` only
- `kast` is a string with `%` — needs cleaning before use
- Some team names have sponsor prefixes (e.g. `Movistar KOI(KOI)`) — handled via `TEAM_NAME_ALIASES` in constants
- All-Star teams present — handled via `INVALID_TEAMS` in constants
- **This is our primary file for player feature engineering**